<a href="https://colab.research.google.com/github/Pooja-V15/NVIDIA-AI-ML-Internship/blob/main/day-8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from diffusers import UNet2DModel, DDPMScheduler

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

# Create U-Net model
model = UNet2DModel(
    sample_size=28,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 128), # Adjusted to 3 stages for 28x28 sample_size
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D"), # Adjusted to 3 blocks
    up_block_types=("UpBlock2D", "UpBlock2D", "UpBlock2D") # Adjusted to 3 blocks
).to(device)

# Diffusion scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

epochs = 5

# Training loop
for epoch in range(epochs):

    for images, _ in train_loader:

        images = images.to(device)

        noise = torch.randn_like(images)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (images.shape[0],),
            device=device
        ).long()

        noisy_images = noise_scheduler.add_noise(
            images,
            noise,
            timesteps
        )

        noise_pred = model(
            noisy_images,
            timesteps
        ).sample

        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss={loss.item():.4f}")

torch.save(model.state_dict(), "mnist_diffusion_model.pth")

print("Training completed.")

Epoch 1/5, Loss=0.0437
